# 🎬 **2D Viral Video Studio (Google Colab Cloud GPU)**
### Run 2D motion graphics with **12 GB RAM + Free NVIDIA GPU**.

**Bas 1 Step:** Niche diye gaye **Play button [ ▶ ]** par click karein!
40 seconds me screen par **Live Link** aayega, uspe click karke prompt daalein aur video banayein!

In [ ]:
# @title 🚀 1-Click Launch 2D Video Studio
import os, sys, time, re, subprocess, urllib.request

print('=' * 55)
print('🚀 Starting 2D Studio on Google Cloud GPU...')
print('=' * 55)

# 1. Download/Update Studio from GitHub
if not os.path.exists('/content/viral-2d-studio'):
    print('📥 Downloading latest studio code...')
    !git clone https://github.com/madhav777-void/viral-2d-studio.git /content/viral-2d-studio
else:
    print('🔄 Pulling updates...')
    !cd /content/viral-2d-studio && git pull

%cd /content/viral-2d-studio

# 2. Install FFmpeg & graphic tools
print('⚙️ Installing FFmpeg and libraries...')
!apt-get update -qq && apt-get install -y -qq ffmpeg fontconfig fonts-dejavu-core > /dev/null
!pip install -q fastapi uvicorn requests numpy scipy pillow skia-python kokoro-onnx onnxruntime pydantic

# 3. Voice model setup
os.makedirs('/content/viral-2d-studio/.omni-shorts/kokoro', exist_ok=True)
onnx_path = '/content/viral-2d-studio/.omni-shorts/kokoro/kokoro-v1.0.onnx'
voices_path = '/content/viral-2d-studio/.omni-shorts/kokoro/voices-v1.0.bin'
if not os.path.exists(onnx_path):
    print('🎙️ Downloading AI Voice Models...')
    urllib.request.urlretrieve('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx', onnx_path)
    urllib.request.urlretrieve('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin', voices_path)

# 4. Cloudflare Tunnel
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('🌐 Setting up public link tunnel...')
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# 5. Environment
os.environ['OMNI_HOME'] = '/content/viral-2d-studio/.omni-shorts'
os.environ['PYTHONPATH'] = '/content/viral-2d-studio/studio:/content/viral-2d-studio/cinematic-skill/skills/cinematic-film/scripts:' + os.environ.get('PYTHONPATH', '')

# 6. Start Studio Server
print('⚡ Starting 2D Video Engine...')
server_proc = subprocess.Popen(['python3', '-m', 'uvicorn', 'server:app', '--app-dir', 'studio', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(3)

# 7. Start Public Tunnel & Print URL
tunnel_proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, text=True)

public_url = None
for line in iter(tunnel_proc.stderr.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('\n' + '🎉 ' * 12)
print(f'👉 YOUR 2D STUDIO LINK: {public_url}')
print('🎉 ' * 12 + '\n')
print('Upar diye gaye link par click karein aur prompt daalkar video banayein!')

try:
    server_proc.wait()
except KeyboardInterrupt:
    server_proc.kill()
    tunnel_proc.kill()
